# Projet de Recherche Opérationnelle

1. Lecture des données depuis fichier ".txt"
2. Affichage des tableaux (coûts, allocations, potentiels, marginaux)
3. Proposition initiale : Nord-Ouest
4. Proposition initiale : Balas-Hammer
5. Calcul du coût total
6. Détection de cycle par BFS
7. Méthode du marche-pied avec potentiel


## Imports

In [1]:
import os, sys, copy
from collections import deque

INF = float('inf')


## 1. Lecture des données
La fonction `lire_fichier` lit un fichier .txt au format donné:


In [2]:
def lire_fichier(chemin):
    """Lit un fichier de problème de transport et retourne (n, m, couts, provisions, commandes)."""
    with open(chemin, 'r') as f:
        lignes = [l.strip() for l in f if l.strip()]
    n, m = map(int, lignes[0].split())
    couts, provisions = [], []
    for i in range(1, n + 1):
        vals = list(map(int, lignes[i].split()))
        couts.append(vals[:m])
        provisions.append(vals[m])
    commandes = list(map(int, lignes[n + 1].split()))
    return n, m, couts, provisions, commandes


### Test de lecture - Problème 5

In [3]:
# Chemin vers le dossier data (adapter si besoin)
DOSSIER_DATA = os.path.join(os.getcwd(), "data")

n, m, couts, provisions, commandes = lire_fichier(os.path.join(DOSSIER_DATA, "probleme5.txt"))
print(f"n={n}, m={m}")
print(f"Coûts       : {couts}")
print(f"Provisions  : {provisions}")
print(f"Commandes   : {commandes}")
print(f"Équilibré   : {sum(provisions) == sum(commandes)}")


n=3, m=3
Coûts       : [[5, 7, 8], [6, 8, 5], [6, 7, 7]]
Provisions  : [25, 25, 25]
Commandes   : [35, 20, 20]
Équilibré   : True


## 2. Fonctions d'affichage


In [4]:
def afficher_matrice_couts(n, m, couts, provisions, commandes):
    """Affiche la matrice des coûts unitaires de transport."""
    print("  MATRICE DES COÛTS")
    print("_"*60)
    all_v = [couts[i][j] for i in range(n) for j in range(m)] + provisions + commandes
    w = max(4, len(str(max(all_v)))) + 2
    lw = max(3, len(f"P{n}")) + 1
    hdr = " "*(lw+2) + "".join(f"C{j+1}".center(w) for j in range(m)) + "  Prov"
    print(hdr)
    print(" "*(lw+2) + "─"*(w*m+6))
    for i in range(n):
        row = f"P{i+1}".ljust(lw+2)
        row += "".join(str(couts[i][j]).center(w) for j in range(m))
        row += f"  {provisions[i]}"
        print(row)
    print(" "*(lw+2) + "─"*(w*m+6))
    print("Cmd".ljust(lw+2) + "".join(str(c).center(w) for c in commandes))


def afficher_proposition(n, m, couts, alloc, provisions, commandes, label="PROPOSITION DE TRANSPORT"):
    """Affiche la table de la proposition de transport (valeurs b[i,j])."""
    print(f"  {label}")
    print("_"*60)
    all_v = [couts[i][j] for i in range(n) for j in range(m)]
    av = [alloc[i][j] for i in range(n) for j in range(m) if alloc[i][j] is not None]
    w = max(5, len(str(max(all_v + (av if av else [0]))))) + 2
    lw = max(3, len(f"P{n}")) + 1
    hdr = " "*(lw+2) + "".join(f"C{j+1}".center(w) for j in range(m)) + "  Prov"
    print(hdr)
    print(" "*(lw+2) + "─"*(w*m+6))
    for i in range(n):
        row = f"P{i+1}".ljust(lw+2)
        for j in range(m):
            v = alloc[i][j]
            row += ("-" if v is None else str(v)).center(w)
        row += f"  {provisions[i]}"
        print(row)
    print(" "*(lw+2) + "─"*(w*m+6))
    print("Cmd".ljust(lw+2) + "".join(str(c).center(w) for c in commandes))


def afficher_table_potentiels(n, m, u, v, couts_pot):
    """Affiche la table u[i]+v[j] (cases hors base)."""
    print("  COÛTS POTENTIELS  (u[i] + v[j])")
    print("_"*60)
    vals = [abs(couts_pot[i][j]) for i in range(n) for j in range(m) if couts_pot[i][j] is not None]
    w = max(8, (len(str(max(vals))) if vals else 1) + 4)
    hdr = "u\\v".center(9) + "".join(
        (f"v{j+1}={v[j]}" if v[j] is not None else f"v{j+1}=?").center(w)
        for j in range(m))
    print(hdr)
    print("─"*(9+w*m))
    for i in range(n):
        su = f"u{i+1}={u[i]}" if u[i] is not None else f"u{i+1}=?"
        row = su.center(9)
        for j in range(m):
            val = couts_pot[i][j]
            row += ("(base)" if val is None else str(val)).center(w)
        print(row)


def afficher_table_marginaux(n, m, couts_marg):
    """Affiche la table c[i,j]-u[i]-v[j] (cases hors base)."""
    print("  COÛTS MARGINAUX  (c[i,j] - u[i] - v[j])")
    print("_"*60)
    vals = [abs(couts_marg[i][j]) for i in range(n) for j in range(m) if couts_marg[i][j] is not None]
    w = max(8, (len(str(max(vals))) if vals else 1) + 4)
    lw = max(3, len(f"P{n}")) + 1
    hdr = " "*(lw+2) + "".join(f"C{j+1}".center(w) for j in range(m))
    print(hdr)
    print(" "*(lw+2) + "─"*(w*m))
    for i in range(n):
        row = f"P{i+1}".ljust(lw+2)
        for j in range(m):
            val = couts_marg[i][j]
            row += ("(base)" if val is None else str(val)).center(w)
        print(row)


### Test d'affichage - Problème 6

In [5]:
n6, m6, c6, p6, cmd6 = lire_fichier(os.path.join(DOSSIER_DATA, "probleme6.txt"))
afficher_matrice_couts(n6, m6, c6, p6, cmd6)


  MATRICE DES COÛTS
____________________________________________________________
        C1    C2    C3    C4    Prov
      ──────────────────────────────
P1      11    12    10    10    60
P2      17    16    15    18    30
P3      19    21    20    22    90
      ──────────────────────────────
Cmd     50    75    30    25  


## 3. Algorithme Nord-Ouest
**Pseudo-code :**
```
i ← 0 ; j ← 0
Tant que i < n et j < m :
    b[i,j] ← min(prov[i], cmd[j])
    prov[i] -= b[i,j] ; cmd[j] -= b[i,j]
    Si prov[i]=0 et cmd[j]=0 : avancer en diagonale (+ arête fictive si dégénéré)
    Sinon si prov[i]=0 : i++
    Sinon : j++
```


In [6]:
def nord_ouest(n, m, provisions, commandes):
    """Proposition initiale par la méthode Nord-Ouest."""
    print("\n" + "─"*60)
    print("  ALGORITHME NORD-OUEST")
    print("─"*60)
    alloc = [[None]*m for _ in range(n)]
    prov, cmd = provisions[:], commandes[:]
    i = j = 0
    while i < n and j < m:
        q = min(prov[i], cmd[j])
        alloc[i][j] = q
        print(f"  b[{i+1},{j+1}] = min({prov[i]}, {cmd[j]}) = {q}")
        prov[i] -= q ; cmd[j] -= q
        if prov[i] == 0 and cmd[j] == 0:
            if i+1 < n and j+1 < m:
                alloc[i+1][j] = 0
                print(f"  (Dégénérescence : arête fictive b[{i+2},{j+1}] = 0)")
            i += 1 ; j += 1
        elif prov[i] == 0: i += 1
        else: j += 1
    return alloc


### Test Nord-Ouest - Problème 5

In [7]:
n5, m5, c5, p5, cmd5 = lire_fichier(os.path.join(DOSSIER_DATA, "probleme5.txt"))
alloc_no5 = nord_ouest(n5, m5, p5, cmd5)
afficher_proposition(n5, m5, c5, alloc_no5, p5, cmd5, "PROPOSITION INITIALE (Nord-Ouest)")



────────────────────────────────────────────────────────────
  ALGORITHME NORD-OUEST
────────────────────────────────────────────────────────────
  b[1,1] = min(25, 35) = 25
  b[2,1] = min(25, 10) = 10
  b[2,2] = min(15, 20) = 15
  b[3,2] = min(25, 5) = 5
  b[3,3] = min(20, 20) = 20
  PROPOSITION INITIALE (Nord-Ouest)
____________________________________________________________
         C1     C2     C3    Prov
      ───────────────────────────
P1       25     -      -     25
P2       10     15     -     25
P3       -      5      20    25
      ───────────────────────────
Cmd      35     20     20  


## 4. Algorithme Balas-Hammer
**Pseudo-code :**
```
Répéter jusqu'à saturation de toutes les lignes/colonnes :
    Calculer la pénalité de chaque ligne/colonne active
        (pénalité = 2e_min - 1er_min)
    Choisir la ligne/colonne de pénalité maximale
    Affecter le max possible sur la case de coût minimum de cette ligne/colonne
    Saturer la ligne ou la colonne correspondante
```


In [8]:
def _penalites(n, m, couts, LA, CA):
    """Calcule les pénalités de Balas-Hammer pour les lignes/colonnes actives."""
    pen_l, pen_c = {}, {}
    for i in LA:
        v = sorted(couts[i][j] for j in CA)
        pen_l[i] = (v[1]-v[0]) if len(v) >= 2 else (v[0] if v else -1)
    for j in CA:
        v = sorted(couts[i][j] for i in LA)
        pen_c[j] = (v[1]-v[0]) if len(v) >= 2 else (v[0] if v else -1)
    return pen_l, pen_c


def balas_hammer(n, m, couts, provisions, commandes):
    """Proposition initiale par la méthode de Balas-Hammer."""
    print("\n" + "─"*60)
    print("  ALGORITHME BALAS-HAMMER")
    print("─"*60)
    alloc = [[None]*m for _ in range(n)]
    prov, cmd = provisions[:], commandes[:]
    LA, CA = set(range(n)), set(range(m))
    it = 0
    while LA and CA:
        it += 1
        pen_l, pen_c = _penalites(n, m, couts, LA, CA)
        print(f"\n  Itération {it}")
        print(f"  Pénalités lignes   : { {f'P{k+1}':v for k,v in pen_l.items()} }")
        print(f"  Pénalités colonnes : { {f'C{k+1}':v for k,v in pen_c.items()} }")
        ml = max(pen_l.values()) if pen_l else -INF
        mc = max(pen_c.values()) if pen_c else -INF
        if ml >= mc:
            bi = max(pen_l, key=lambda x: pen_l[x])
            bj = min(CA, key=lambda j: couts[bi][j])
            print(f"  → Pén. max = {ml} sur ligne P{bi+1}")
            print(f"  → Coût min sur P{bi+1} : c[{bi+1},{bj+1}] = {couts[bi][bj]}")
        else:
            bj = max(pen_c, key=lambda x: pen_c[x])
            bi = min(LA, key=lambda i: couts[i][bj])
            print(f"  → Pén. max = {mc} sur col. C{bj+1}")
            print(f"  → Coût min sur C{bj+1} : c[{bi+1},{bj+1}] = {couts[bi][bj]}")
        q = min(prov[bi], cmd[bj])
        alloc[bi][bj] = q
        print(f"  → b[{bi+1},{bj+1}] = min({prov[bi]}, {cmd[bj]}) = {q}")
        prov[bi] -= q ; cmd[bj] -= q
        if prov[bi] == 0 and cmd[bj] == 0:
            LA.discard(bi) ; CA.discard(bj)
        elif prov[bi] == 0: LA.discard(bi)
        else: CA.discard(bj)
    return alloc


### Test Balas-Hammer - Problème 5

In [9]:
alloc_bh5 = balas_hammer(n5, m5, c5, p5, cmd5)
afficher_proposition(n5, m5, c5, alloc_bh5, p5, cmd5, "PROPOSITION INITIALE (Balas-Hammer)")



────────────────────────────────────────────────────────────
  ALGORITHME BALAS-HAMMER
────────────────────────────────────────────────────────────

  Itération 1
  Pénalités lignes   : {'P1': 2, 'P2': 1, 'P3': 1}
  Pénalités colonnes : {'C1': 1, 'C2': 0, 'C3': 2}
  → Pén. max = 2 sur ligne P1
  → Coût min sur P1 : c[1,1] = 5
  → b[1,1] = min(25, 35) = 25

  Itération 2
  Pénalités lignes   : {'P2': 1, 'P3': 1}
  Pénalités colonnes : {'C1': 0, 'C2': 1, 'C3': 2}
  → Pén. max = 2 sur col. C3
  → Coût min sur C3 : c[2,3] = 5
  → b[2,3] = min(25, 20) = 20

  Itération 3
  Pénalités lignes   : {'P2': 2, 'P3': 1}
  Pénalités colonnes : {'C1': 0, 'C2': 1}
  → Pén. max = 2 sur ligne P2
  → Coût min sur P2 : c[2,1] = 6
  → b[2,1] = min(5, 10) = 5

  Itération 4
  Pénalités lignes   : {'P3': 1}
  Pénalités colonnes : {'C1': 6, 'C2': 7}
  → Pén. max = 7 sur col. C2
  → Coût min sur C2 : c[3,2] = 7
  → b[3,2] = min(25, 20) = 20

  Itération 5
  Pénalités lignes   : {'P3': 6}
  Pénalités colonnes 

## 5. Calcul du coût total

In [10]:
def cout_total(n, m, couts, alloc):
    """Calcule le coût total Σ c[i,j] * b[i,j]."""
    return sum(couts[i][j] * alloc[i][j]
               for i in range(n) for j in range(m)
               if alloc[i][j] is not None and alloc[i][j] > 0)

print("Coût NO  :", cout_total(n5, m5, c5, alloc_no5))
print("Coût BH  :", cout_total(n5, m5, c5, alloc_bh5))


Coût NO  : 480
Coût BH  : 425


## 6. Graphe biparti et parcours BFS
Le graphe de transport est un graphe biparti :
- Nœuds `0..n-1` → fournisseurs  
- Nœuds `n..n+m-1` → clients  
- Une arête `(i, n+j)` existe ssi `b[i,j]` est dans la base.

**Détection de cycle** : BFS classique ; si on atteint un sommet déjà visité différent du parent → cycle.  
**Détection de connexité** : BFS depuis tous les nœuds actifs ; si plusieurs composantes → non connexe.


In [11]:
def base(n, m, alloc):
    """Retourne la liste des arêtes de base (i,j)."""
    return [(i,j) for i in range(n) for j in range(m) if alloc[i][j] is not None]

def adj_graph(n, m, alloc):
    """Construit la liste d'adjacence du graphe biparti."""
    g = {k: [] for k in range(n+m)}
    for i, j in base(n, m, alloc):
        g[i].append(n+j)
        g[n+j].append(i)
    return g

def _rec_cycle(u, v, par):
    """Reconstitue le cycle à partir d'une arête de retour (u→v)."""
    def path_root(x):
        p = []
        while x != -1: p.append(x) ; x = par[x]
        return p
    pu, pv = path_root(u), path_root(v)
    set_pu = {nd: k for k, nd in enumerate(pu)}
    for k2, nd in enumerate(pv):
        if nd in set_pu:
            k1 = set_pu[nd]
            return pu[:k1+1] + pv[k2-1::-1]
    return pu + pv[::-1]

def detecter_cycle_bfs(n, m, alloc):
    """Détecte un cycle par BFS. Retourne (True, cycle) ou (False, [])."""
    g = adj_graph(n, m, alloc)
    vis, par = {}, {}
    for s in range(n+m):
        if s in vis or not g[s]: continue
        q = deque([s]) ; vis[s] = True ; par[s] = -1
        while q:
            u = q.popleft()
            for v in g[u]:
                if v not in vis:
                    vis[v] = True ; par[v] = u ; q.append(v)
                elif par[u] != v:
                    return True, _rec_cycle(u, v, par)
    return False, []

def est_connexe_bfs(n, m, alloc):
    """Teste la connexité par BFS. Retourne (True, []) ou (False, composantes)."""
    g = adj_graph(n, m, alloc)
    actifs = set()
    for i, j in base(n, m, alloc):
        actifs.add(i) ; actifs.add(n+j)
    if not actifs: return True, []
    vis, comps = set(), []
    for s in sorted(actifs):
        if s in vis: continue
        comp = [] ; q = deque([s]) ; vis.add(s)
        while q:
            u = q.popleft() ; comp.append(u)
            for v in g[u]:
                if v not in vis and v in actifs:
                    vis.add(v) ; q.append(v)
        comps.append(comp)
    return len(comps) <= 1, comps

def afficher_cycle(cycle, n):
    noms = [f"P{nd+1}" if nd < n else f"C{nd-n+1}" for nd in cycle]
    print("  Cycle : " + " → ".join(noms) + f" → {noms[0]}")


### Test BFS - cycle et connexité

In [12]:
# Tester sur une proposition initiale (Nord-Ouest pb 5)
a_cycle, cycle = detecter_cycle_bfs(n5, m5, alloc_no5)
print("Cycle :", a_cycle, "->", cycle if a_cycle else "aucun")
connexe, comps = est_connexe_bfs(n5, m5, alloc_no5)
print("Connexe :", connexe, "/ Nb composantes :", len(comps))


Cycle : False -> aucun
Connexe : True / Nb composantes : 1


## 7. Boucle fermée 
Pour ajouter l'arête améliorante (ei, ej), on cherche un chemin alterné
qui revient à (ei, ej) en n'empruntant que les arêtes de la base courante.


In [13]:
def trouver_boucle_fermee(n, m, alloc, ei, ej):
    """
    Cherche la boucle fermée passant par (ei, ej) dans la base étendue.
    Retourne une liste de cellules ou [] si introuvable.
    """
    aretes = set(base(n, m, alloc)) | {(ei, ej)}
    by_row = {i: [] for i in range(n)}
    by_col = {j: [] for j in range(m)}
    for r, c in aretes:
        by_row[r].append(c)
        by_col[c].append(r)

    def dfs(path, move):
        ri, ci = path[-1]
        if len(path) >= 4:
            if move == 'row' and ci == ej and ri == ei: return path[:]
            if move == 'col' and ri == ei and ci == ej: return path[:]
        if move == 'col':          # se déplacer sur la ligne ri
            for nc in by_row[ri]:
                if nc == ci: continue
                nxt = (ri, nc)
                if nxt not in set(path[1:]):
                    path.append(nxt)
                    r = dfs(path, 'row')
                    if r: return r
                    path.pop()
        else:                      # se déplacer sur la colonne ci
            for nr in by_col[ci]:
                if nr == ri: continue
                nxt = (nr, ci)
                if nxt not in set(path[1:]):
                    path.append(nxt)
                    r = dfs(path, 'col')
                    if r: return r
                    path.pop()
        return None

    for first_move in ('col', 'row'):
        r = dfs([(ei, ej)], first_move)
        if r: return r
    return []


## 8. Maximisation sur une boucle
**Pseudo-code :**
```
Affecter signes + / - en alternant sur la boucle (arête entrante = +)
δ ← min des b[i,j] avec signe -
Mettre à jour : b[i,j] ± δ pour chaque case de la boucle
Supprimer les cases tombées à 0 (hors arête entrante)
```


In [14]:
def maximiser_boucle(n, m, alloc, boucle, arete_entrante):
    """
    Maximise le transport sur une boucle fermée.
    boucle : liste de cellules (i,j), la première est l'arête entrante.
    Retourne (nouvelle_alloc, aretes_supprimees).
    """
    ei, ej = arete_entrante
    signes = {cell: ('+' if k % 2 == 0 else '-') for k, cell in enumerate(boucle)}
    print("\n  Conditions sur les cases de la boucle :")
    for ri, ci in boucle:
        s = signes[(ri, ci)]
        v = alloc[ri][ci] if alloc[ri][ci] is not None else 0
        print(f"    b[{ri+1},{ci+1}] {s} δ  (valeur actuelle : {v})")
    negs = [(ri, ci) for (ri, ci), s in signes.items() if s == '-']
    if not negs:
        print("  Aucune arête à soustraire.")
        return alloc, []
    delta = min(alloc[ri][ci] if alloc[ri][ci] is not None else 0 for ri, ci in negs)
    print(f"\n  δ = {delta}")
    na = copy.deepcopy(alloc)
    if na[ei][ej] is None: na[ei][ej] = 0
    supprimees = []
    for (ri, ci), s in signes.items():
        cur = na[ri][ci] if na[ri][ci] is not None else 0
        if s == '+':
            na[ri][ci] = cur + delta
        else:
            na[ri][ci] = cur - delta
            if na[ri][ci] == 0 and (ri, ci) != (ei, ej):
                supprimees.append((ri, ci))
    for ri, ci in supprimees: na[ri][ci] = None
    if supprimees:
        print("  Arête(s) supprimée(s) : " + ", ".join(f"b[{r+1},{c+1}]" for r,c in supprimees))
    else:
        print("  δ = 0 : aucune modification (cas dégénéré).")
    return na, supprimees


## 9. Normalisation de la proposition
Une proposition est **non dégénérée** (base admissible) si elle est à la fois :
- **Acyclique** : aucun cycle dans le graphe biparti
- **Connexe** : le graphe est un arbre couvrant

Nombre d'arêtes de base requis : `n + m - 1`


In [15]:
def rendre_acyclique(n, m, couts, alloc, verbose=True):
    """Élimine tous les cycles en maximisant sur chacun d'eux."""
    for _ in range(300):
        ok, cycle = detecter_cycle_bfs(n, m, alloc)
        if not ok: break
        if verbose:
            print("\n Cycle détecté :")
            afficher_cycle(cycle, n)
        boucle = []
        nodes = cycle + [cycle[0]]
        for k in range(len(nodes) - 1):
            u, v = nodes[k], nodes[k+1]
            if u < n and v >= n: boucle.append((u, v-n))
            elif v < n and u >= n: boucle.append((v, u-n))
        if boucle:
            alloc, _ = maximiser_boucle(n, m, alloc, boucle, boucle[0])
    return alloc


def rendre_connexe(n, m, couts, alloc, verbose=True):
    """Rend la proposition connexe en ajoutant des arêtes fictives (coût croissant)."""
    for _ in range(n + m):
        ok, comps = est_connexe_bfs(n, m, alloc)
        if ok: break
        if verbose:
            print(f"\n  Non connexe ({len(comps)} composantes).")
            for k, comp in enumerate(comps):
                noms = [f"P{nd+1}" if nd < n else f"C{nd-n+1}" for nd in comp]
                print(f"    Comp.{k+1}: {', '.join(noms)}")
        comp_of = {nd: k for k, comp in enumerate(comps) for nd in comp}
        best, bc = None, INF
        for i in range(n):
            for j in range(m):
                ci, cj = comp_of.get(i, -1), comp_of.get(n+j, -2)
                if ci != cj and alloc[i][j] is None and couts[i][j] < bc:
                    bc = couts[i][j] ; best = (i, j)
        if best is None: print("  Impossible de relier !") ; break
        bi, bj = best ; alloc[bi][bj] = 0
        if verbose: print(f"  → Arête fictive b[{bi+1},{bj+1}]=0 (coût {couts[bi][bj]})")
    return alloc


def normaliser(n, m, couts, alloc, verbose=True):
    """Rend la proposition acyclique et connexe (arbre couvrant valide)."""
    alloc = rendre_acyclique(n, m, couts, alloc, verbose)
    alloc = rendre_connexe(n, m, couts, alloc, verbose)
    alloc = rendre_acyclique(n, m, couts, alloc, verbose)
    nb = sum(1 for i in range(n) for j in range(m) if alloc[i][j] is not None)
    attendu = n + m - 1
    if nb != attendu and verbose:
        print(f"  Arêtes de base : {nb} (attendu {attendu})")
    return alloc


## 10. Calcul des potentiels
On résout le système u[i] + v[j] = c[i,j] pour chaque arête de base,
avec la convention u[0] = 0, par propagation itérative.


In [16]:
def calculer_potentiels(n, m, couts, alloc):
    """Calcule u[i] et v[j] par propagation (u[0]=0 fixé)."""
    u, v = [None]*n, [None]*m
    u[0] = 0
    for _ in range((n+m)*2):
        changed = False
        for i in range(n):
            for j in range(m):
                if alloc[i][j] is not None:
                    if u[i] is not None and v[j] is None:
                        v[j] = couts[i][j] - u[i] ; changed = True
                    elif v[j] is not None and u[i] is None:
                        u[i] = couts[i][j] - v[j] ; changed = True
        if not changed: break
    return u, v


def couts_potentiels(n, m, u, v, alloc):
    """u[i]+v[j] pour les cases hors base (None pour les cases de base)."""
    cp = [[None]*m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            if alloc[i][j] is None and u[i] is not None and v[j] is not None:
                cp[i][j] = u[i] + v[j]
    return cp


def couts_marginaux(n, m, couts, u, v, alloc):
    """c[i,j]-u[i]-v[j] pour les cases hors base (None pour les cases de base)."""
    cm = [[None]*m for _ in range(n)]
    for i in range(n):
        for j in range(m):
            if alloc[i][j] is None and u[i] is not None and v[j] is not None:
                cm[i][j] = couts[i][j] - u[i] - v[j]
    return cm


## 11. Méthode du marche-pied avec potentiel
**Structure générale :**
```
Répéter :
  1. Afficher proposition courante + coût
  2. Normaliser si dégénérée
  3. Calculer potentiels u[i], v[j]
  4. Calculer coûts marginaux c[i,j] - u[i] - v[j]
  5. Si tous ≥ 0 → OPTIMAL, sortir
  6. Sinon : trouver l'arête de coût marginal le plus négatif
  7. Trouver la boucle fermée la contenant
  8. Maximiser le transport sur cette boucle
```


In [17]:
def marche_pied(n, m, couts, alloc_init, provisions, commandes):
    """
    Méthode du marche-pied avec potentiel.
    Retourne (alloc_optimale, cout_optimal).
    """
    print("  MÉTHODE DU MARCHE-PIED AVEC POTENTIEL")
    print("_"*60)
    alloc = copy.deepcopy(alloc_init)
    MAX_ITER = 500

    for iteration in range(1, MAX_ITER+1):
        print(f"\n{'─'*60}")
        print(f"  Itération {iteration}")
        print(f"{'─'*60}")
        afficher_proposition(n, m, couts, alloc, provisions, commandes,
                             f"PROPOSITION COURANTE (itération {iteration})")
        cout = cout_total(n, m, couts, alloc)
        print(f"\n  Coût total : {cout}")

        # Vérification dégénérescence
        nb = sum(1 for i in range(n) for j in range(m) if alloc[i][j] is not None)
        attendu = n + m - 1
        if nb != attendu:
            print(f"\n  Dégénérée ({nb}/{attendu}) donc normalisation.")
            alloc = normaliser(n, m, couts, alloc)

        # Cycle résiduel
        ok, cycle = detecter_cycle_bfs(n, m, alloc)
        if ok:
            print("\n  Cycle résiduel :")
            afficher_cycle(cycle, n)
            alloc = rendre_acyclique(n, m, couts, alloc)

        # Potentiels
        u, v = calculer_potentiels(n, m, couts, alloc)
        print("\n  Potentiels :")
        print("  " + "  ".join(f"u{i+1}={u[i]}" for i in range(n)))
        print("  " + "  ".join(f"v{j+1}={v[j]}" for j in range(m)))

        cp = couts_potentiels(n, m, u, v, alloc)
        cm = couts_marginaux(n, m, couts, u, v, alloc)
        afficher_table_potentiels(n, m, u, v, cp)
        afficher_table_marginaux(n, m, cm)

        # Arête améliorante
        best_val, best_arc = 0, None
        for i in range(n):
            for j in range(m):
                if cm[i][j] is not None and cm[i][j] < best_val:
                    best_val = cm[i][j] ; best_arc = (i, j)

        if best_arc is None:
            print("\n  Tous les coûts marginaux sont ≥ 0 donc solution OPTIMALE.")
            break

        ai, aj = best_arc
        print(f"\n  → Arête améliorante : b[{ai+1},{aj+1}] (coût marginal = {best_val})")

        boucle = trouver_boucle_fermee(n, m, alloc, ai, aj)
        if not boucle:
            print("  Boucle introuvable → ajout fictif et normalisation.")
            alloc[ai][aj] = 0
            alloc = normaliser(n, m, couts, alloc)
            continue

        print("  Boucle fermée : " +
              " → ".join(f"({r+1},{c+1})" for r, c in boucle) +
              f" → ({boucle[0][0]+1},{boucle[0][1]+1})")
        alloc, _ = maximiser_boucle(n, m, alloc, boucle, (ai, aj))

    else:
        print(f"\n  Arrêt après {MAX_ITER} itérations.")

    cout_final = cout_total(n, m, couts, alloc)
    afficher_proposition(n, m, couts, alloc, provisions, commandes, "PROPOSITION OPTIMALE")
    print(f"\n  Coût optimal : {cout_final}")
    return alloc, cout_final


## 12. Programme interactif complet
Il demande :
1. Le numéro du problème (1–12)
2. L'algorithme initial (Nord-Ouest ou Balas-Hammer)

Entrez `q` pour quitter.


In [18]:
def resoudre_probleme(num, algo):
    """
    Résout le problème numéro 'num' avec l'algorithme 'algo' ('NO' ou 'BH').
    """
    chemin = os.path.join(DOSSIER_DATA, f"probleme{num}.txt")
    n, m, couts, provisions, commandes = lire_fichier(chemin)
    print(f"\nProblème {num} : {n} fournisseur(s) × {m} client(s).")
    afficher_matrice_couts(n, m, couts, provisions, commandes)

    if algo.upper() == 'NO':
        alloc_init = nord_ouest(n, m, provisions, commandes)
        label = "NO"
    else:
        alloc_init = balas_hammer(n, m, couts, provisions, commandes)
        label = "BH"

    afficher_proposition(n, m, couts, alloc_init, provisions, commandes,
                         f"PROPOSITION INITIALE ({label})")
    c0 = cout_total(n, m, couts, alloc_init)
    print(f"\nCoût initial ({label}) : {c0}")

    alloc_norm = normaliser(n, m, couts, copy.deepcopy(alloc_init))
    _, cout_opt = marche_pied(n, m, couts, alloc_norm, provisions, commandes)

    gain = c0 - cout_opt
    print(f"\n─── Résumé ───")
    print(f"Coût initial ({label}) : {c0}")
    print(f"Coût optimal          : {cout_opt}")
    print(f"Gain                  : {gain}" + (f" ({gain/c0*100:.1f}%)" if c0 > 0 else ""))
    return cout_opt


### Exemple : Problème 5 avec Nord-Ouest

In [19]:
resoudre_probleme(5, 'NO')



Problème 5 : 3 fournisseur(s) × 3 client(s).
  MATRICE DES COÛTS
____________________________________________________________
        C1    C2    C3    Prov
      ────────────────────────
P1      5     7     8     25
P2      6     8     5     25
P3      6     7     7     25
      ────────────────────────
Cmd     35    20    20  

────────────────────────────────────────────────────────────
  ALGORITHME NORD-OUEST
────────────────────────────────────────────────────────────
  b[1,1] = min(25, 35) = 25
  b[2,1] = min(25, 10) = 10
  b[2,2] = min(15, 20) = 15
  b[3,2] = min(25, 5) = 5
  b[3,3] = min(20, 20) = 20
  PROPOSITION INITIALE (NO)
____________________________________________________________
         C1     C2     C3    Prov
      ───────────────────────────
P1       25     -      -     25
P2       10     15     -     25
P3       -      5      20    25
      ───────────────────────────
Cmd      35     20     20  

Coût initial (NO) : 480
  MÉTHODE DU MARCHE-PIED AVEC POTENTIEL
___

425

### Exemple  : Problème 6 avec Balas-Hammer

In [27]:
resoudre_probleme(6, 'BH')



Problème 6 : 3 fournisseur(s) × 4 client(s).
  MATRICE DES COÛTS
____________________________________________________________
        C1    C2    C3    C4    Prov
      ──────────────────────────────
P1      11    12    10    10    60
P2      17    16    15    18    30
P3      19    21    20    22    90
      ──────────────────────────────
Cmd     50    75    30    25  

────────────────────────────────────────────────────────────
  ALGORITHME BALAS-HAMMER
────────────────────────────────────────────────────────────

  Itération 1
  Pénalités lignes   : {'P1': 0, 'P2': 1, 'P3': 1}
  Pénalités colonnes : {'C1': 6, 'C2': 4, 'C3': 5, 'C4': 8}
  → Pén. max = 8 sur col. C4
  → Coût min sur C4 : c[1,4] = 10
  → b[1,4] = min(60, 25) = 25

  Itération 2
  Pénalités lignes   : {'P1': 1, 'P2': 1, 'P3': 1}
  Pénalités colonnes : {'C1': 6, 'C2': 4, 'C3': 5}
  → Pén. max = 6 sur col. C1
  → Coût min sur C1 : c[1,1] = 11
  → b[1,1] = min(35, 50) = 35

  Itération 3
  Pénalités lignes   : {'P2': 1, 

2880

### Boucle interactive 


In [20]:
def main_interactif():
    print("__________________________________________________________")
    print("      SOLVEUR DE PROBLÈMES DE TRANSPORT                   ")
    print("__________________________________________________________")
    while True:
        print("  MENU – numéro du problème (1–12) ou 'q'")
        print("_"*60)
        choix = input("  > ").strip()
        if choix.lower() == 'q':
            print("\n  Byee !") ; break
        try:
            num = int(choix) ; assert 1 <= num <= 12
        except:
            print("  Numéro invalide.") ; continue
        print("\n  [1] Nord-Ouest   [2] Balas-Hammer")
        a = input("  Choix (1/2) : ").strip()
        algo = 'NO' if a == '1' else 'BH'
        resoudre_probleme(num, algo)

main_interactif()   


__________________________________________________________
      SOLVEUR DE PROBLÈMES DE TRANSPORT                   
__________________________________________________________
  MENU – numéro du problème (1–12) ou 'q'
____________________________________________________________
  > 1

  [1] Nord-Ouest   [2] Balas-Hammer
  Choix (1/2) : 1

Problème 1 : 2 fournisseur(s) × 2 client(s).
  MATRICE DES COÛTS
____________________________________________________________
        C1    C2    Prov
      ──────────────────
P1      30    20    100
P2      10    50    100
      ──────────────────
Cmd    100   100  

────────────────────────────────────────────────────────────
  ALGORITHME NORD-OUEST
────────────────────────────────────────────────────────────
  b[1,1] = min(100, 100) = 100
  (Dégénérescence : arête fictive b[2,1] = 0)
  b[2,2] = min(100, 100) = 100
  PROPOSITION INITIALE (NO)
____________________________________________________________
         C1     C2    Prov
      ─────────────

## 13. Récapitulatif de tous les problèmes

In [21]:
import io, contextlib

def solve_silent(num, algo):
    """Résout silencieusement et retourne le coût optimal."""
    chemin = os.path.join(DOSSIER_DATA, f"probleme{num}.txt")
    n_, m_, c_, p_, cmd_ = lire_fichier(chemin)
    if algo == 'NO':
        a = nord_ouest(n_, m_, p_[:], cmd_[:])
    else:
        a = balas_hammer(n_, m_, c_, p_[:], cmd_[:])
    c0 = cout_total(n_, m_, c_, a)
    a = normaliser(n_, m_, c_, copy.deepcopy(a), verbose=False)
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        _, opt = marche_pied(n_, m_, c_, a, p_, cmd_)
    return c0, opt

print(f"{'Pb':>3} | {'Init NO':>9} | {'Opt NO':>9} | {'Init BH':>9} | {'Opt BH':>9}")
print("─" * 55)
for num in range(1, 13):
    c0_no, opt_no = solve_silent(num, 'NO')
    c0_bh, opt_bh = solve_silent(num, 'BH')
    print(f"{num:>3} | {c0_no:>9} | {opt_no:>9} | {c0_bh:>9} | {opt_bh:>9}")


 Pb |   Init NO |    Opt NO |   Init BH |    Opt BH
───────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  ALGORITHME NORD-OUEST
────────────────────────────────────────────────────────────
  b[1,1] = min(100, 100) = 100
  (Dégénérescence : arête fictive b[2,1] = 0)
  b[2,2] = min(100, 100) = 100

────────────────────────────────────────────────────────────
  ALGORITHME BALAS-HAMMER
────────────────────────────────────────────────────────────

  Itération 1
  Pénalités lignes   : {'P1': 10, 'P2': 40}
  Pénalités colonnes : {'C1': 20, 'C2': 30}
  → Pén. max = 40 sur ligne P2
  → Coût min sur P2 : c[2,1] = 10
  → b[2,1] = min(100, 100) = 100

  Itération 2
  Pénalités lignes   : {'P1': 20}
  Pénalités colonnes : {'C2': 20}
  → Pén. max = 20 sur ligne P1
  → Coût min sur P1 : c[1,2] = 20
  → b[1,2] = min(100, 100) = 100
  1 |      8000 |      3000 |      3000 |      3000

─────────────────────────────────────────────────────